# 第49章 回归图（regplot / lmplot）

通过regplot和lmplot叠加回归趋势，检查线性关系和分组差异。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

需要描述两个数值变量的拟合方向，而不是证明因果。

## 数据结构

两列数值；lmplot可增加分类分组和分面。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 order=1 改为 order=2，观察线性与二次多项式拟合的曲线差异
2. 修改 ci=95 为 ci=None，对比显示与隐藏置信区间的视觉效果
3. 添加 robust=True 参数，说明稳健拟合对异常点的抗性


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(data=marketing, x="visits", y="sales", scatter_kws={"alpha": 0.45, "s": 25}, line_kws={"color": "#d93025"}, ax=ax)
ax.set(title="访问量与销售额线性趋势", xlabel="访问量", ylabel="销售额")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
grid = sns.lmplot(data=marketing, x="visits", y="sales", hue="channel", col="channel", col_wrap=3, height=3.2, scatter_kws={"alpha": 0.4, "s": 22}, palette="colorblind")
grid.set_axis_labels("访问量", "销售额")
grid.set_titles("{col_name}")
grid.fig.suptitle("分渠道回归趋势", y=1.04)
plt.show()


## 3. 参数说明

- order：多项式阶数
- robust：稳健拟合
- ci：区间
- scatter_kws/line_kws：样式


## 4. 结果解读

读取斜率方向、散点离散和区间；检查异常点是否主导拟合。


## 常见误区

- 把回归线解释为因果
- 忽略非线性和异方差
- 只展示拟合线不展示原始点


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(data=marketing, x="ad_spend", y="sales", order=2, scatter_kws={"alpha": 0.4, "s": 24}, line_kws={"color": "#188038"}, ax=ax)
ax.set(title="广告投入与销售额的非线性趋势检查", xlabel="广告投入", ylabel="销售额")
fig.tight_layout()
plt.show()


## 本章小结

通过regplot和lmplot叠加回归趋势，检查线性关系和分组差异。


### 你已经掌握

- 判断回归图（regplot / lmplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 需要描述两个数值变量的拟合方向，而不是证明因果。 |
| 数据结构 | 两列数值；lmplot可增加分类分组和分面。 |
| 结果解读 | 读取斜率方向、散点离散和区间；检查异常点是否主导拟合。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `order` | 多项式阶数 |
| `robust` | 稳健拟合 |
| `ci` | 区间 |
| `scatter_kws/line_kws` | 样式 |


### 需要注意

- 把回归线解释为因果
- 忽略非线性和异方差
- 只展示拟合线不展示原始点


### 完成检查

- [ ] 能判断什么问题适合使用回归图（regplot / lmplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
